# Identify artifact data

In [89]:
import numpy as np
import pandas as pd
import warnings
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

warnings.filterwarnings("ignore", category=FutureWarning)

dataset = BNCI2014_001(artifact_handling="annotate")
paradigm = MotorImagery(n_classes=4, fmin=None, fmax=None)

X, labels, meta = paradigm.get_data(dataset=dataset)

results = []

for subject in range(1, 10):
    raw_data = dataset.get_data(subjects=[subject])

    for session, runs in raw_data[subject].items():
        for run_name, raw in runs.items():

            # Find global MOABB indices for this run
            mask = (
                (meta["subject"].astype(str) == str(subject)) &
                (meta["session"].astype(str) == str(session)) &
                (meta["run"].astype(str) == str(run_name))
            )

            run_indices = np.where(mask.to_numpy())[0]

            for ann in raw.annotations:
                if ann["description"] == "bnci_artifact":

                    trial_idx = int(ann["extras"]["trial"]) - 1
                    moabb_idx = run_indices[trial_idx]

                    results.append({
                        "trial_index": moabb_idx,
                        "subject": subject,
                        "session": session,
                        "run": run_name,
                        "class": labels[moabb_idx]
                    })

# Save results
df_artifacts = pd.DataFrame(results)
df_artifacts = df_artifacts.sort_values("trial_index")

df_artifacts.to_csv(r"C:\Users\pahan\OneDrive\Documents\Fall 2026\BMEN 600\Project\bmen-600-group-11\Results\trial_artifacts", index=False)


display(df_artifacts)

Choosing from all possible events


,trial_index,subject,session,run,class
0,21,1,0train,0,tongue
1,45,1,0train,0,right_hand
2,52,1,0train,1,left_hand
3,56,1,0train,1,right_hand
4,67,1,0train,1,feet
...,...,...,...,...,...
483,5164,9,1test,5,right_hand
484,5166,9,1test,5,tongue
485,5169,9,1test,5,left_hand
486,5170,9,1test,5,left_hand


### Load Train Data

In [ ]:
import os
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery


def load_train_data(
    subjects=None,
    t_start=0,
    t_end=4,
    low_pass=40,
    high_pass=4,
    f_resample=None,
    shuffle=False,
    random_state=None,
    save_as_np=False,
    save_dir=None,
):
    """
    Load and preprocess training EEG data from BNCI2014_001 using MOABB.

    Parameters
    ----------
    subjects : list or None
        Subject IDs to load. None loads all subjects.
    t_start : float
        Start time of the epoch in seconds.
    t_end : float
        End time of the epoch in seconds.
    low_pass : float
        Upper cutoff frequency (Hz).
    high_pass : float
        Lower cutoff frequency (Hz).
    f_resample : float or None
        Target sampling frequency (Hz).
    shuffle : bool
        Whether to shuffle trials.
    random_state : int or None
        Random seed for reproducible shuffling.
    save_as_np : bool
        Whether to save processed data as NumPy files.
    save_dir : str or None
        Directory for saving processed data.

    Returns
    -------
    X : np.ndarray
        EEG data (trials, channels, samples).
    y : np.ndarray
        Class labels.
    metadata : pd.DataFrame
        Trial metadata.
    """

    # 1. Validate parameters
    if t_start >= t_end:
        raise ValueError("t_start must be smaller than t_end.")

    if high_pass is None or low_pass is None:
        raise ValueError(
            "Both high_pass and low_pass must be specified."
        )

    if high_pass >= low_pass:
        raise ValueError(
            "high_pass must be smaller than low_pass."
        )

    # 2. Initialize dataset
    dataset = BNCI2014_001()

    if subjects is None:
        subjects = dataset.subject_list

    # 3. Configure preprocessing
    paradigm = MotorImagery(
        n_classes=4,
        fmin=high_pass,
        fmax=low_pass,
        tmin=t_start,
        tmax=t_end,
        resample=f_resample
    )

    # 4. Load and preprocess data
    X, y, metadata = paradigm.get_data(
        dataset=dataset,
        subjects=subjects
    )

    # 5. Select training session only
    train_mask = (metadata["session"] == "0train").to_numpy()

    X_train = X[train_mask]
    y_train = y[train_mask]
    metadata_train = metadata.iloc[np.flatnonzero(train_mask)].reset_index(drop=True)

    # 6. Shuffle trials if requested
    if shuffle:
        rng = np.random.default_rng(random_state)
        indices = rng.permutation(len(X_train))

        X_train = X_train[indices]
        y_train = y_train[indices]
        metadata_train = metadata_train.iloc[indices].reset_index(drop=True)

    # 7. Save processed data if requested
    if save_as_np:
        if save_dir is None:
            save_dir = "processed_train_data"

        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, "X_train.npy"), X_train)
        np.save(os.path.join(save_dir, "y_train.npy"), y_train)

        metadata_train.to_csv(
            os.path.join(save_dir, "metadata_train.csv"),
            index=False
        )

        print(f"Processed data saved to: {save_dir}")

    return X_train, y_train, metadata_train

### Load validation_test data

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery


def load_validation_test_data(
    subjects=None,
    t_start=0,
    t_end=4,
    low_pass=40,
    high_pass=4,
    f_resample=None,
    validation_test_ratio=0.8,
    shuffle=False,
    random_state=None,
    save_as_np=False,
    save_dir=None,
):
    """
    Load and preprocess evaluation EEG data from BNCI2014_001
    and split it into validation and test datasets.

    Parameters
    ----------
    subjects : list or None
        Subject IDs to load. None loads all subjects.
    t_start : float
        Start time of the epoch in seconds.
    t_end : float
        End time of the epoch in seconds.
    low_pass : float
        Upper cutoff frequency (Hz).
    high_pass : float
        Lower cutoff frequency (Hz).
    f_resample : float or None
        Target sampling frequency (Hz).
    validation_test_ratio : float
        Proportion of evaluation data used for validation.
    shuffle : bool
        Whether to shuffle trials before splitting.
    random_state : int or None
        Random seed for reproducibility.
    save_as_np : bool
        Whether to save processed data as NumPy files.
    save_dir : str or None
        Directory for saving processed data.

    Returns
    -------
    X_val, y_val, metadata_val :
        Validation EEG data, labels and metadata.
    X_test, y_test, metadata_test :
        Test EEG data, labels and metadata.
    """

    # 1. Validate parameters
    if t_start >= t_end:
        raise ValueError("t_start must be smaller than t_end.")

    if high_pass is None or low_pass is None:
        raise ValueError(
            "Both high_pass and low_pass must be specified."
        )

    if high_pass >= low_pass:
        raise ValueError(
            "high_pass must be smaller than low_pass."
        )

    if not 0 < validation_test_ratio < 1:
        raise ValueError(
            "validation_test_ratio must be between 0 and 1."
        )

    # 2. Initialize dataset
    dataset = BNCI2014_001()

    if subjects is None:
        subjects = dataset.subject_list

    # 3. Configure preprocessing
    paradigm = MotorImagery(
        n_classes=4,
        fmin=high_pass,
        fmax=low_pass,
        tmin=t_start,
        tmax=t_end,
        resample=f_resample
    )

    # 4. Load and preprocess data
    X, y, metadata = paradigm.get_data(
        dataset=dataset,
        subjects=subjects
    )

    # 5. Select evaluation session only
    test_mask = (metadata["session"] == "1test").to_numpy()

    X = X[test_mask]
    y = y[test_mask]
    metadata = metadata.iloc[
        np.flatnonzero(test_mask)
    ].reset_index(drop=True)

    if len(X) == 0:
        raise ValueError("No evaluation trials found.")

    # 6. Split into validation and test sets
    # Stratify by subject and class to preserve both distributions
    stratify_labels = (
        metadata["subject"].astype(str).to_numpy()
        + "_"
        + y.astype(str)
    )

    indices = np.arange(len(X))

    val_indices, test_indices = train_test_split(
        indices,
        train_size=validation_test_ratio,
        random_state=random_state,
        shuffle=True,
        stratify=stratify_labels
    )

    X_val = X[val_indices]
    y_val = y[val_indices]
    metadata_val = metadata.iloc[
        val_indices
    ].reset_index(drop=True)

    X_test = X[test_indices]
    y_test = y[test_indices]
    metadata_test = metadata.iloc[
        test_indices
    ].reset_index(drop=True)

    # 7. Optional shuffling within each split
    if shuffle:
        rng = np.random.default_rng(random_state)

        val_order = rng.permutation(len(X_val))
        test_order = rng.permutation(len(X_test))

        X_val = X_val[val_order]
        y_val = y_val[val_order]
        metadata_val = metadata_val.iloc[
            val_order
        ].reset_index(drop=True)

        X_test = X_test[test_order]
        y_test = y_test[test_order]
        metadata_test = metadata_test.iloc[
            test_order
        ].reset_index(drop=True)

    # 8. Save processed data if requested
    if save_as_np:
        if save_dir is None:
            save_dir = "processed_validation_test_data"

        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, "X_val.npy"), X_val)
        np.save(os.path.join(save_dir, "y_val.npy"), y_val)
        np.save(os.path.join(save_dir, "X_test.npy"), X_test)
        np.save(os.path.join(save_dir, "y_test.npy"), y_test)

        metadata_val.to_csv(
            os.path.join(save_dir, "metadata_val.csv"),
            index=False
        )

        metadata_test.to_csv(
            os.path.join(save_dir, "metadata_test.csv"),
            index=False
        )

        print(f"Processed data saved to: {save_dir}")

    return (
        X_val, y_val, metadata_val,
        X_test, y_test, metadata_test
    )